In [ ]:
import subprocess

result = subprocess.run(
    ["ip", "netns", "exec", "web01", "nft", "list", "ruleset"],
    capture_output=True,
    text=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

In [ ]:
import json
import subprocess
from datetime import datetime, timezone

def collect_ruleset(namespace="web01"):
    result = subprocess.run(
        [
            "ip", "netns", "exec", namespace,
            "nft", "-j", "list", "ruleset",
        ],
        capture_output=True,
        text=True,
        timeout=15,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"nft collection failed (exit {result.returncode}): "
            f"{result.stderr.strip()}"
        )

    try:
        payload = json.loads(result.stdout)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"nft returned invalid JSON: {exc}"
        ) from exc

    validate_ruleset(payload)

    return {
        "collected_at": datetime.now(timezone.utc).isoformat(),
        "namespace": namespace,
        "ruleset": payload,
    }

In [ ]:
from typing import Any
from pydantic import BaseModel, ConfigDict, Field


class NftObject(BaseModel):
    model_config = ConfigDict(extra="allow")


class NftRule(NftObject):
    family: str
    table: str
    chain: str
    expr: list[dict[str, Any]]
    handle: int | None = None


class NftChain(NftObject):
    family: str
    table: str
    name: str
    type: str | None = None
    hook: str | None = None
    prio: int | str | None = None
    policy: str | None = None
    handle: int | None = None


class NftTable(NftObject):
    family: str
    name: str
    handle: int | None = None


def validate_ruleset(payload: dict) -> None:
    if not isinstance(payload, dict):
        raise ValueError("Ruleset root must be a JSON object")

    objects = payload.get("nftables")

    if not isinstance(objects, list):
        raise ValueError("'nftables' must be a list")

    supported = {
        "metainfo": None,
        "table": NftTable,
        "chain": NftChain,
        "rule": NftRule,
    }

    for index, item in enumerate(objects):
        if not isinstance(item, dict) or len(item) != 1:
            raise ValueError(
                f"Invalid nftables entry at index {index}"
            )

        kind, value = next(iter(item.items()))

        if kind not in supported:
            # Keep sets, maps, counters, flowtables, etc.
            # They may be needed for later visualizations.
            continue

        model = supported[kind]
        if model is not None:
            model.model_validate(value)

        elif not isinstance(value, dict):
            raise ValueError(
                f"Invalid metainfo at index {index}"
            )

In [ ]:
import pandas as pd
import json


def rules_to_dataframe(payload: dict) -> pd.DataFrame:
    rows = []

    for item in payload["nftables"]:
        rule = item.get("rule")
        if rule is None:
            continue

        counters = []
        verdicts = []
        jumps = []
        gotos = []

        for expr in rule.get("expr", []):
            if not isinstance(expr, dict):
                continue

            if "counter" in expr:
                counters.append(expr["counter"])

            if "jump" in expr:
                jumps.append(expr["jump"])

            if "goto" in expr:
                gotos.append(expr["goto"])

            if "accept" in expr:
                verdicts.append("accept")
            if "drop" in expr:
                verdicts.append("drop")
            if "reject" in expr:
                verdicts.append("reject")
            if "return" in expr:
                verdicts.append("return")

        packets = sum(
            c.get("packets", 0)
            for c in counters
            if isinstance(c, dict)
        )
        bytes_ = sum(
            c.get("bytes", 0)
            for c in counters
            if isinstance(c, dict)
        )

        rows.append({
            "family": rule.get("family"),
            "table": rule.get("table"),
            "chain": rule.get("chain"),
            "handle": rule.get("handle"),
            "packets": packets if counters else None,
            "bytes": bytes_ if counters else None,
            "verdicts": ", ".join(verdicts),
            "jumps": ", ".join(
                j.get("target", "")
                for j in jumps if isinstance(j, dict)
            ),
            "gotos": ", ".join(
                g.get("target", "")
                for g in gotos if isinstance(g, dict)
            ),
            "expressions": json.dumps(
                rule.get("expr", []), sort_keys=True
            ),
        })

    return pd.DataFrame(rows)

In [ ]:
snapshot = collect_ruleset("web01")
rules_df = rules_to_dataframe(snapshot["ruleset"])

print("Collected:", snapshot["collected_at"])
print("Namespace:", snapshot["namespace"])
print("Rules:", len(rules_df))

display(rules_df)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

family_filter = widgets.Dropdown(
    options=["All"] + sorted(
        rules_df["family"].dropna().unique().tolist()
    ),
    description="Family:",
)

table_filter = widgets.Dropdown(
    options=["All"] + sorted(
        rules_df["table"].dropna().unique().tolist()
    ),
    description="Table:",
)

chain_filter = widgets.Text(
    description="Chain:",
    placeholder="Filter by chain name",
)

output = widgets.Output()


def render_rules(*_):
    df = rules_df

    if family_filter.value != "All":
        df = df[df["family"] == family_filter.value]

    if table_filter.value != "All":
        df = df[df["table"] == table_filter.value]

    if chain_filter.value:
        df = df[
            df["chain"].str.contains(
                chain_filter.value,
                case=False,
                na=False,
                regex=False,
            )
        ]

    with output:
        clear_output(wait=True)
        display(df.reset_index(drop=True))


for control in (family_filter, table_filter, chain_filter):
    control.observe(render_rules, names="value")

display(widgets.VBox([
    family_filter,
    table_filter,
    chain_filter,
    output,
]))

render_rules()